# NYC Mobility - Source Inspection

## Why we inspect first

We check the source first so we know what we are working with before we create tables. These checks describe the raw files only; they do not clean or change any records.


## Green Taxi

The required monthly Parquet files cover March, April, and May 2026. We start with a temporary March view so the source can be inspected without creating a Bronze object.

### March preview


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW src_green_taxi_2026_03 AS
SELECT *
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
    format => 'parquet'
);

The temporary view points to the landed March file and exists only for this notebook session.


In [0]:
%sql
SELECT *
FROM src_green_taxi_2026_03
LIMIT 20;

### Schema

We inspect the inferred source columns and data types before designing later layers.


In [0]:
%sql
DESCRIBE src_green_taxi_2026_03;

### March row count

The original run recorded **44,208 source rows** for March.


In [0]:
%sql
SELECT COUNT(*) AS march_source_row_count
FROM src_green_taxi_2026_03;

### Basic source quality checks

We check timestamps, location IDs, negative numeric values, and rescued data. These are observations only—Bronze must still keep the rows as received.


In [0]:
%sql
WITH source AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    COUNT(*) AS total_rows,

    MIN(lpep_pickup_datetime) AS earliest_pickup,
    MAX(lpep_pickup_datetime) AS latest_pickup,

    SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
    SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,

    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,

    SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,

    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,

    SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows

FROM source;

### Observation

March has no null pickup/drop-off timestamps or location IDs and no rescued rows. We did observe one invalid time order, 111 negative fares, 113 negative totals, and an earliest pickup in 2009.

### Why it matters

These records may need a business rule later, but changing them now would stop Bronze from being raw.

### Decision

Keep every row in Bronze and revisit the unusual records in Silver.


### Observed date overlap and unusual records

This check shows which pickup year and month appear inside the March file.


In [0]:
%sql
WITH source AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    YEAR(lpep_pickup_datetime) AS pickup_year,
    MONTH(lpep_pickup_datetime) AS pickup_month,
    COUNT(*) AS row_count
FROM source
GROUP BY
    YEAR(lpep_pickup_datetime),
    MONTH(lpep_pickup_datetime)
ORDER BY
    pickup_year,
    pickup_month;

The March file contains 44,199 March 2026 rows, eight February 2026 rows, and one January 2009 row. We record the overlap instead of filtering it here.


### March-May monthly profiling

We run the same basic checks across all three landed files.


In [0]:
%sql
-- profile all three monthly source files before bronze

WITH march AS (
    SELECT
        '2026-03' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
),

april AS (
    SELECT
        '2026-04' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )
),

may AS (
    SELECT
        '2026-05' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
)

SELECT * FROM march
UNION ALL
SELECT * FROM april
UNION ALL
SELECT * FROM may
ORDER BY source_month;

The source counts are **44,208 for March**, **44,238 for April**, and **44,921 for May**. Negative fare and total values remain present, while the inspected key timestamps and location IDs have no nulls.


In [0]:
%sql
WITH all_taxi AS (
    SELECT '2026-03' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT '2026-04' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT '2026-05' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
)

SELECT
    source_month,
    YEAR(lpep_pickup_datetime) AS pickup_year,
    MONTH(lpep_pickup_datetime) AS pickup_month,
    COUNT(*) AS row_count
FROM all_taxi
GROUP BY
    source_month,
    YEAR(lpep_pickup_datetime),
    MONTH(lpep_pickup_datetime)
ORDER BY
    source_month,
    pickup_year,
    pickup_month;

### Observation

The file labels and pickup months do not align perfectly. April includes one March and two May pickups; May includes two December 2008 and eight April pickups.

### Decision

Treat the filename as ingestion provenance and preserve the source timestamps. Any date-quality rule belongs in Silver.


## Taxi Zones

We inspect the reference CSV before using it to interpret pickup and drop-off IDs.


In [0]:
%sql
-- temporary source inspection only
CREATE OR REPLACE TEMP VIEW src_taxi_zones AS

SELECT *
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
    format => 'csv',
    header => true
);

### Preview


In [0]:
%sql
SELECT *
FROM src_taxi_zones
LIMIT 20;

### Schema


In [0]:
%sql
DESCRIBE src_taxi_zones;

### Row count

The original inspection recorded **265 Taxi Zone rows**.


In [0]:
%sql
SELECT COUNT(*) AS taxi_zone_row_count
FROM src_taxi_zones;

### LocationID uniqueness

An empty result means no duplicate `LocationID` was found.


In [0]:
%sql
SELECT
    LocationID,
    COUNT(*) AS record_count
FROM src_taxi_zones
GROUP BY LocationID
HAVING COUNT(*) > 1;

### Required-field null checks

The original result shows zero nulls in `LocationID`, `Borough`, `Zone`, and `service_zone`.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) AS null_location_id,
    SUM(CASE WHEN Borough IS NULL THEN 1 ELSE 0 END) AS null_borough,
    SUM(CASE WHEN Zone IS NULL THEN 1 ELSE 0 END) AS null_zone,
    SUM(CASE WHEN service_zone IS NULL THEN 1 ELSE 0 END) AS null_service_zone
FROM src_taxi_zones;

### Pickup relationship check

An empty result means every March pickup location matched the lookup.


In [0]:
%sql
-- check pickup location ids that do not exist in the taxi zone lookup

WITH taxi AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    t.PULocationID,
    COUNT(*) AS trip_count
FROM taxi t
LEFT JOIN src_taxi_zones z
    ON t.PULocationID = z.LocationID
WHERE z.LocationID IS NULL
GROUP BY t.PULocationID
ORDER BY trip_count DESC;

### Drop-off relationship check

An empty result means every March drop-off location matched the lookup.


In [0]:
%sql
-- check dropoff location ids that do not exist in the taxi zone lookup

WITH taxi AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    t.DOLocationID,
    COUNT(*) AS trip_count
FROM taxi t
LEFT JOIN src_taxi_zones z
    ON t.DOLocationID = z.LocationID
WHERE z.LocationID IS NULL
GROUP BY t.DOLocationID
ORDER BY trip_count DESC;

We found 265 Taxi Zones, no duplicate `LocationID`, no required-field nulls, and no unmatched March pickup or drop-off IDs. This is verified source evidence. The Bronze implementation is now prepared in the Bronze notebooks, but its new results remain pending until those cells are run in Databricks.


## Weather

Run the ingestion notebook first so the raw Open-Meteo JSON exists in R2. Here we only inspect the saved file; we do not flatten or clean it.

### Request and saved-file metadata

- Source system: `open_meteo`
- API: Open-Meteo Archive API
- Requested period: `2026-03-01` through `2026-05-31`
- Requested timezone: `America/New_York`
- Exact saved file: `open_meteo_2026-03-01_2026-05-31.json`
- Verified ingestion response: HTTP 200
- API Source Link: https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006&start_date=2026-03-01&end_date=2026-05-31&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m&timezone=America%2FNew_York 


In [0]:
%python
from pathlib import Path

weather_file = Path(
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather/"
    "open_meteo_2026-03-01_2026-05-31.json"
)

print("Exists:", weather_file.exists())
print("Size:", weather_file.stat().st_size, "bytes")

### JSON structure and metadata


In [0]:
%python
import json

weather_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather/"
    "open_meteo_2026-03-01_2026-05-31.json"
)

with open(weather_path, "r") as file:
    weather = json.load(file)

print("Top-level keys:")
print(weather.keys())

print("\nLocation metadata:")
print("Latitude:", weather.get("latitude"))
print("Longitude:", weather.get("longitude"))
print("Timezone:", weather.get("timezone"))
print("UTC offset seconds:", weather.get("utc_offset_seconds"))

print("\nHourly variables:")
print(weather["hourly"].keys())

times = weather["hourly"]["time"]

print("\nHourly record check:")
print("Number of timestamps:", len(times))
print("First timestamp:", times[0])
print("Last timestamp:", times[-1])

The request returned HTTP 200 during ingestion. The saved response contains 2,208 hourly timestamps from `2026-03-01T00:00` through `2026-05-31T23:00`. The API returned nearby grid coordinates, which we retain as source metadata.


In [0]:
hourly = weather["hourly"]

for column_name, values in hourly.items():
    print(column_name, len(values))

### Hourly-array null checks

Each inspected hourly array has 2,208 values.


In [0]:
for column_name, values in hourly.items():
    null_count = sum(value is None for value in values)

    print(
        column_name,
        "| records:", len(values),
        "| nulls:", null_count
    )

The existing output shows zero nulls for time, temperature, precipitation, rain, snowfall, weather code, and wind speed. The 2,208 observations remain inside the raw JSON. The one-record Bronze implementation is prepared, but it has not been executed or validated in Databricks yet.


## Traffic Advisory

This is our optional web-scraping source. The original run saved one raw HTML file and one metadata JSON file. We point to those verified files for inspection.


In [0]:
from pathlib import Path

html_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory/"
    "nyc_dot_weekend_traffic_20260914T040846Z.html"
)
metadata_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory/"
    "nyc_dot_weekend_traffic_20260914T040846Z.metadata.json"
)


### Saved-file confirmation


In [0]:
from pathlib import Path

print("HTML exists:", Path(html_path).exists())
print("Metadata exists:", Path(metadata_path).exists())

print("HTML size:", Path(html_path).stat().st_size, "bytes")
print("Metadata size:", Path(metadata_path).stat().st_size, "bytes")

### Raw HTML preview


In [0]:
with open(html_path, "r", encoding="utf-8", errors="replace") as file:
    preview = file.read(1000)

print(preview)

### Page title and headings


In [0]:
from bs4 import BeautifulSoup

with open(html_path, "r", encoding="utf-8", errors="replace") as file:
    soup = BeautifulSoup(file, "html.parser")

print("Page title:")
print(soup.title.get_text(" ", strip=True) if soup.title else "No title found")

print("\nHeadings found:")
for heading in soup.find_all(["h1", "h2", "h3", "h4"]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(heading.name, "|", text)

### Body-text inspection


In [0]:
page_text = soup.get_text("\n", strip=True)

print(page_text[:5000])

### Table check

The current page contains no HTML tables, so any later parser must rely on the observed heading and paragraph structure.


In [0]:
tables = soup.find_all("table")

print("Number of HTML tables:", len(tables))

### Keyword checks


In [0]:
keywords = [
    "September",
    "2026",
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Staten Island"
]

for keyword in keywords:
    print(keyword, "->", keyword.lower() in page_text.lower())

### Event-heading structure


In [0]:
# inspect how advisory/event headings are structured in the raw html

event_headings = soup.find_all("h4")

print("Number of h4 event headings:", len(event_headings))

for heading in event_headings[:10]:
    print("\nEVENT:")
    print(heading.get_text(" ", strip=True))

    next_element = heading.find_next_sibling()

    if next_element:
        print("NEXT TAG:", next_element.name)
        print(
            "DETAIL:",
            next_element.get_text(" ", strip=True)[:500]
        )
    else:
        print("No next sibling found")

### Observation

The saved advisory describes September 2026 events, while the Taxi analytical period is March-May 2026.

### Why it matters

Joining these records would be temporally incorrect.

### Decision

Keep this as a bonus ingestion and source-inspection demonstration only. Do not integrate it into March-May analytics.


## Source Inspection Summary

- Green Taxi: three monthly Parquet files profiled; counts verified; unusual dates and negative amounts documented for later Silver decisions.
- Taxi Zones: 265 rows; unique `LocationID`; required fields complete; March pickup and drop-off IDs matched.
- Weather: raw JSON saved and structurally inspected; 2,208 hourly values per field with zero observed nulls.
- Traffic Advisory: raw HTML and metadata saved and inspected as a September 2026 bonus example.

Green Taxi already has executed Bronze evidence. The Taxi Zones, Weather, and bonus Traffic Advisory Bronze implementations are prepared in the next notebooks. Their results must be produced in Databricks before we call them validated.
